# Chatterbox Nepali TTS — Cloud Inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Firojpaudel/chatterbox-nepali/blob/feature/vll-inference/Chatterbox_NP_run.ipynb)

This notebook provides a cloud-based environment for the Chatterbox Nepali TTS model. It utilizes NVIDIA T4 GPUs to perform high-fidelity speech synthesis from Devanagari text.

---

## Step 0: GPU Check
Before starting, ensure that you have a GPU enabled.

**Go to: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**

In [ ]:
# @title Check GPU Availability
import torch

if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    !nvidia-smi
else:
    print("❌ ERROR: No GPU detected! Synthesis will be extremely slow on CPU.")
    print("Please go to Runtime -> Change runtime type and select T4 GPU.")

## Step 1: Environment Setup
We use `uv` for high-performance dependency resolution and `hf_transfer` for accelerated model downloads.

In [ ]:
# @title Install Dependencies & Speed Boost
import os

print("🚀 Installing uv...")
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = f"{os.environ['HOME']}/.local/bin:{os.environ['PATH']}"

print("📦 Cleaning and Cloning repository...")
!rm -rf /content/chatterbox-nepali
!git clone -b feature/vll-inference https://github.com/Firojpaudel/chatterbox-nepali.git
%cd /content/chatterbox-nepali

print("🛠️ Creating isolated virtual environment...")
!uv venv .venv
VENV_PATH = "/content/chatterbox-nepali/.venv/bin/python"
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = "1"

print("⚙️ Installing core project and dependencies into .venv...")
!uv pip install --python $VENV_PATH --no-cache -e .

print("🎙️ Installing accelerators and audio libraries into .venv...")
!uv pip install --python $VENV_PATH --no-cache hf_transfer perth librosa safetensors huggingface_hub diffusers torchaudio pykakasi g2p-en vllm==0.9.2 peft llvmlite phonemizer

print("✅ Setup Complete!")

## Step 2: Launch Interface
Choose your inference mode below:

| Mode | Speed | VRAM | Notes |
|------|-------|------|-------|
| **Standard** | ~15s/chunk | ~10 GB | Full model with CFG, chunking, hallucination protection |
| **vLLM** | ~3-5s/chunk | ~3 GB | GPU-accelerated inference via vLLM engine |

> **Note**: Please wait $\sim 6 \text{ mins}$ once you start the gradio server. Click on the public link and let the models download first in the first run.

In [ ]:
# @title 🎙️ Launch: Standard Mode (Full features, ~10GB VRAM)
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = "1"
# Standard mode: loads the full Chatterbox model with all features
# You can toggle vLLM on/off from the Gradio UI checkbox
!MPLBACKEND=Agg ./.venv/bin/python gradio_app.py

In [ ]:
# @title ⚡ Launch: vLLM Mode (Fast inference)
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = "1"
os.environ['ENABLE_VLLM'] = "1"
# vLLM mode: skips loading the standard model, loads vLLM engine at startup
# This avoids Gradio timeout issues and uses much less VRAM
!MPLBACKEND=Agg ./.venv/bin/python gradio_app.py

### Technical Notes
- **GPU**: Ensure T4 GPU is selected for real-time synthesis speeds.
- **hf_transfer**: Enabled for accelerated model downloads (up to 10x faster).
- **uv**: This notebook uses the `uv` package manager for significantly faster and more reliable installations.
- **vLLM Mode**: Set `ENABLE_VLLM=1` to pre-load vLLM at startup. This prevents Gradio timeout issues and uses only ~3GB VRAM instead of ~10GB.
- **Standard vs vLLM**: Standard mode offers full features (CFG, hallucination protection). vLLM mode is faster but runs in SAFE_MODE without CFG batch doubling.

---

### Links
- [GitHub Repository](https://github.com/Firojpaudel/chatterbox-nepali)
- [Hugging Face Model Weights](https://huggingface.co/Firoj112/chatterbox-nepali-runs)